In [1]:
from pathlib import Path

import pandas as pd

from methanehunter.config import load_config
from methanehunter.data.metadata import find_unseen_sites, load_metadata

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
cfg = load_config(ROOT / "configs/data.yaml")
csv_path = ROOT / cfg["paths"]["metadata_dir"] / cfg["dataset"]["metadata_filename"]
df = load_metadata(csv_path, cfg["columns"]["required"])
print(df.shape)

d:\Github\MethaneHunter\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(93538, 45)


In [2]:
print(df["split_name"].value_counts(dropna=False))
df.groupby("split_name")["isplume"].agg(["count", "sum", "mean"])

split_name
test_2023     43524
train_2023    38345
val_2023       6018
Not Used       5651
Name: count, dtype: int64


,count,sum,mean
split_name,,,
Not Used,5651,410,0.072554
test_2023,43524,1832,0.042092
train_2023,38345,3512,0.091590
val_2023,6018,299,0.049684


In [3]:
print(pd.crosstab(df["satellite"], df["split_name"]))
print(df.groupby("sector")["isplume"].agg(["count", "mean"]))
print(df.groupby("offshore")["isplume"].agg(["count", "mean"]))

split_name  Not Used  test_2023  train_2023  val_2023
satellite                                            
LC08            1156       9149        3729       237
LC09             870       8937        2954        25
S2A             1953      12555       15780      2870
S2B             1671      12726       15882      2886
S2C                1        157           0         0
                     count      mean
sector                              
Coal                   222  0.000000
Controlled Releases   1569  0.009560
Oil and Gas          91241  0.066176
Waste                  109  0.000000
unknown                397  0.000000
          count      mean
offshore                 
False     90305  0.065234
True       3233  0.050108


In [4]:
print("Duplicate image ids:", df["id_loc_image"].duplicated().sum())
print(df.groupby("split_name")["tile_date"].agg(["min", "max"]))
df.isna().mean().sort_values(ascending=False).head(10)

Duplicate image ids: 0
                                     min                           max
split_name                                                            
Not Used   2013-03-19 10:08:53.238+00:00 2024-12-29 18:06:49.000+00:00
test_2023  2024-01-01 00:05:22.713+00:00 2024-12-31 17:27:23.051+00:00
train_2023 2018-01-01 09:13:51.000+00:00 2023-11-30 10:33:09.000+00:00
val_2023   2021-01-01 07:03:09.000+00:00 2021-12-31 10:03:23.429+00:00


plumepath                0.935288
ch4path                  0.935288
window_col_off           0.935117
window_width             0.935117
window_height            0.935117
window_row_off           0.935117
background_image_tile    0.032842
sza                      0.000000
vza                      0.000000
wind_v                   0.000000
dtype: float64

In [5]:
print(pd.crosstab(df["ch4path"].notna(), df["isplume"], rownames=["has_ch4path"]))
pd.crosstab(df["observability"], df["isplume"])

isplume      False  True 
has_ch4path              
False        87485      0
True             0   6053


isplume,False,True
observability,,
bad_retrieval,2619,13
clear,80698,6040
cloudy,2991,0
shadow,636,0
smoke,137,0
snow,404,0


In [6]:
s = cfg["splits"]
unseen = find_unseen_sites(df, s["train"], s["test"])
test_df = df[df["split_name"] == s["test"]]
print("Unseen test sites:", len(unseen))
print("Test rows from unseen sites:", test_df["id_location"].isin(unseen).sum(), "/", len(test_df))

Unseen test sites: 697
Test rows from unseen sites: 15655 / 43524
